# Editor de Audio por Texto

Editor de audio mediante edicion de texto. Corre en GPU de Colab.

## Como usar
1. Ejecuta todas las celdas de instalacion una vez
2. Subi un archivo de audio (wav, mp3, etc.)
3. La transcripcion se genera automaticamente
4. En la terminal de Colab (icono >_), ejecuta:
   ```
   python audio_editor_tui.py
   ```
5. Edita el texto, borra palabras para silenciarlas, reordena
6. Presiona Ctrl+P para escuchar el preview
7. Presiona Ctrl+R para renderizar el audio editado
8. Descarga el resultado


In [ ]:
# Celda 1: Instalar dependencias
!pip install -q textual faster-whisper
!apt-get install -qq ffmpeg
print("Dependencias instaladas")

## Subir audio
Ejecuta la siguiente celda y selecciona un archivo de audio.

In [ ]:
# Celda 2: Subir archivo de audio
from google.colab import files
uploaded = files.upload()
audio_path = list(uploaded.keys())[0]
import subprocess, os
# Convertir a wav si es necesario
if not audio_path.lower().endswith('.wav'):
    wav_path = audio_path.rsplit('.', 1)[0] + '.wav'
    subprocess.run(['ffmpeg', '-y', '-i', audio_path, '-acodec', 'pcm_s16le', '-ar', '16000', wav_path],
                   check=True, capture_output=True)
    audio_path = wav_path
# Renombrar a nombre fijo
os.rename(audio_path, 'input_audio.wav')
print(f"Audio listo: input_audio.wav ({os.path.getsize('input_audio.wav')//1024}KB)")

## Transcripcion con GPU
Usamos faster-whisper en la GPU T4 de Colab.

In [ ]:
# Celda 3: Transcribir audio
from faster_whisper import WhisperModel
import json, time

print("Cargando modelo tiny en GPU...")
model = WhisperModel("tiny", device="cuda", compute_type="float16")

print("Transcribiendo...")
t0 = time.time()
segs, info = model.transcribe("input_audio.wav", word_timestamps=True, beam_size=1, vad_filter=True)

words = []
wid = 0
for s in segs:
    for w in (s.words or []):
        words.append({"id": wid, "text": w.word.strip(), "start": round(w.start,2), "end": round(w.end,2)})
        wid += 1

result = {"duration": round(info.duration, 2), "words": words, "elapsed": round(time.time() - t0, 2)}

with open("words.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print(f"Transcripcion: {len(words)} palabras en {result['elapsed']}s")
print(f"Duracion audio: {result['duration']}s")

## Generar TUI
Se crea el archivo `audio_editor_tui.py` con el editor interactivo.

In [ ]:
# Celda 4: Escribir la TUI
tui_code = '''# audio_editor_tui.py - TUI para editar audio por texto
# Se genera automaticamente dentro de Colab.

import json, subprocess, tempfile, time, os, hashlib, shutil
from pathlib import Path
from textual.app import App, ComposeResult
from textual.widgets import Header, Footer, Static, TextArea, Button, Label, Input
from textual.containers import Horizontal, Vertical, ScrollableContainer
from textual.screen import ModalScreen
from textual.binding import Binding
from textual import events
from textual.reactive import reactive

AUDIO_PATH = "input_audio.wav"
WORDS_PATH = "words.json"

# ── Cargar datos ──────────────────────────────────────────────────

def load_data():
    with open(WORDS_PATH, encoding="utf-8") as f:
        return json.load(f)  # {duration, words[{id,text,start,end}]}

data = load_data()
original_words = data["words"]
duration = data["duration"]


# ── Utilidades de audio ───────────────────────────────────────────

def get_edited_audio_path(edited_text: str) -> str | None:
    """Renderiza el audio editado a un archivo temp y devuelve la ruta."""
    ids = compute_surviving_ids(original_words, edited_text)
    if not ids:
        return None
    segs = []
    for sid in ids:
        w = next((w for w in original_words if w["id"] == sid), None)
        if w:
            segs.append((w["start"], w["end"]))
    if not segs:
        return None
    
    segfile = tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False, encoding="utf-8")
    try:
        for start, end in segs:
            if end - start <= 0:
                continue
            segfile.write(f"file '{Path(AUDIO_PATH).resolve()}'\n")
            segfile.write(f"inpoint {start}\n")
            segfile.write(f"outpoint {end}\n")
        segfile.close()
        out = tempfile.mktemp(suffix=".wav")
        subprocess.run(
            ["ffmpeg", "-y", "-f", "concat", "-safe", "0",
             "-i", segfile.name, "-c", "copy", out],
            check=True, capture_output=True, text=True
        )
        return out
    finally:
        Path(segfile.name).unlink(missing_ok=True)


def play_audio(path: str):
    """Reproduce audio en background con ffplay."""
    subprocess.Popen(
        ["ffplay", "-nodisp", "-autoexit", "-loglevel", "quiet", path],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )


# ── Alineación de texto ───────────────────────────────────────────

def compute_surviving_ids(original_words, edited_text):
    edited = edited_text.strip().lower().split()
    if not edited:
        return []
    text_to_ids = {}
    for w in original_words:
        k = w["text"].lower()
        text_to_ids.setdefault(k, []).append(w["id"])
    used = set()
    result = []
    for ew in edited:
        q = text_to_ids.get(ew)
        if q:
            for sid in q:
                if sid not in used:
                    used.add(sid)
                    result.append(sid)
                    break
    return result


# ── Widgets personalizados ────────────────────────────────────────

class WordChips(ScrollableContainer):
    """Muestra palabras como chips clickeables."""
    
    def __init__(self, words, **kwargs):
        super().__init__(**kwargs)
        self.words = words
        self.edited_text = ""
    
    def compose(self):
        for w in self.words:
            yield WordChip(w, classes="chip")
    
    def update_chips(self, edited_text: str):
        self.edited_text = edited_text
        edited_words = set(edited_text.strip().lower().split())
        for chip in self.query(WordChip):
            in_text = chip.word["text"].lower() in edited_words
            chip.set_class(not in_text, "deleted")
            chip.set_class(in_text, "present")


class WordChip(Static):
    """Un chip de palabra individual."""
    
    def __init__(self, word, **kwargs):
        super().__init__(word["text"], **kwargs)
        self.word = word
    
    def on_click(self, event):
        if event.button == 3:  # right click
            self.app.push_screen(ContextMenu(self.word))
        else:
            self.app.notify(f"Click en: {self.word['text']} ({self.word['start']}s - {self.word['end']}s)")


class ContextMenu(ModalScreen):
    """Menu contextual para acciones por palabra."""
    
    def __init__(self, word, **kwargs):
        super().__init__(**kwargs)
        self.word = word
    
    def compose(self):
        with Vertical(id="menu"):
            yield Static(f"Palabra: [bold]{self.word['text']}[/bold]", id="menu-title")
            yield Button("Borrar palabra", id="delete", variant="error")
            yield Button("Silenciar 0.5s", id="silence")
            yield Button("Separar aqui", id="split")
            yield Button("Cancelar", id="cancel")
    
    def on_button_pressed(self, event):
        if event.button.id == "cancel":
            self.app.pop_screen()
            return
        self.app.pop_screen()
        editor = self.app.query_one("#editor", TextArea)
        w = self.word
        if event.button.id == "delete":
            editor.text = editor.text.replace(w["text"], "", 1).replace("  ", " ")
        elif event.button.id == "silence":
            editor.text = editor.text.replace(w["text"], "[silencio]", 1)
        elif event.button.id == "split":
            editor.text = editor.text.replace(w["text"], w["text"] + "\n\n", 1)
        self.app.sync_chips()


# ── Pantalla principal ────────────────────────────────────────────

class EditorScreen(Static):
    """Pantalla principal del editor."""
    
    def compose(self):
        yield Static(f"Audio: {AUDIO_PATH} | Duracion: {duration:.1f}s", id="file-info")
        with Horizontal(id="toolbar"):
            yield Button("> Play preview", id="play", variant="primary")
            yield Button("Renderizar", id="render", variant="default")
            yield Button("Descargar", id="download", variant="default")
            yield Button("Salir", id="quit", variant="error")
        yield WordChips(original_words, id="chips")
        yield TextArea(id="editor", soft_wrap=True, show_line_numbers=False)
        yield Static(id="status")
    
    def on_mount(self):
        editor = self.query_one("#editor", TextArea)
        editor.text = " ".join(w["text"] for w in original_words)
        self.sync_chips()
    
    def sync_chips(self):
        editor = self.query_one("#editor", TextArea)
        chips = self.query_one("#chips", WordChips)
        chips.update_chips(editor.text)
    
    def on_button_pressed(self, event):
        if event.button.id == "play":
            self.app.action_preview()
        elif event.button.id == "render":
            self.app.action_render()
        elif event.button.id == "download":
            self.app.action_download()
        elif event.button.id == "quit":
            self.app.action_quit()


class AudioEditorApp(App):
    """TUI Editor de Audio por Texto."""
    
    CSS = """
    Screen {
        background: #1a1a2e;
    }
    
    #file-info {
        padding: 1 2;
        color: #8af;
        text-style: bold;
    }
    
    #toolbar {
        margin: 0 1;
        height: 3;
    }
    #toolbar Button {
        margin: 0 1 0 0;
        min-width: 14;
    }
    
    #chips {
        height: 7;
        border: solid #3d3d5e;
        margin: 1 1;
        padding: 1;
        overflow-y: auto;
    }
    .chip {
        padding: 0 1;
        margin: 0 1 1 0;
        background: #2d2d5e;
        min-width: 2;
        height: 1;
    }
    .chip:hover {
        background: #4d4d8e;
    }
    .chip.deleted {
        background: #3a1a1a;
        text-style: strikethrough;
        color: #666;
    }
    .chip.present {
        background: #2d2d5e;
        color: #e0e0e0;
    }
    
    #editor {
        margin: 0 1;
        border: solid #3d3d5e;
        height: 10;
    }
    #editor:focus {
        border: solid #5a5a9e;
    }
    
    #status {
        padding: 1 2;
        color: #888;
    }
    
    #menu {
        background: #22224a;
        border: thick #3d3d6e;
        padding: 1;
        width: 40;
        margin: 8 12;
    }
    #menu-title {
        padding: 0 0 1 0;
        text-style: bold;
    }
    #menu Button {
        margin: 0 0 1 0;
    }
    """
    
    BINDINGS = [
        Binding("ctrl+r", "render", "Renderizar"),
        Binding("ctrl+p", "preview", "Preview"),
        Binding("ctrl+q", "quit", "Salir"),
        Binding("escape", "close_menu", "Cerrar menu"),
    ]
    
    def compose(self):
        yield Header()
        yield EditorScreen()
        yield Footer()
    
    def on_mount(self):
        self.title = "Editor de Audio por Texto"
        self.sub_title = "Editá el texto para editar el audio"
    
    def sync_chips(self):
        self.query_one(EditorScreen).sync_chips()
    
    # ── Acciones ──────────────────────────────────────────────────
    
    last_rendered = None
    
    def action_render(self):
        editor = self.query_one("#editor", TextArea)
        status = self.query_one("#status")
        status.update("Renderizando audio...")
        out = get_edited_audio_path(editor.text)
        if out:
            dur = Path(out).stat().st_size
            status.update(f"Audio renderizado: {os.path.basename(out)} ({dur//1000}KB)")
            self.last_rendered = out
            self.notify("Audio renderizado!")
        else:
            status.update("Error: no hay palabras para renderizar")
    
    def action_preview(self):
        editor = self.query_one("#editor", TextArea)
        status = self.query_one("#status")
        status.update("Preparando preview...")
        out = get_edited_audio_path(editor.text)
        if out:
            play_audio(out)
            status.update(f"Reproduciendo preview...")
            self.last_rendered = out
        else:
            status.update("Error: no hay palabras para reproducir")
    
    def action_download(self):
        from google.colab import files
        if self.last_rendered:
            files.download(self.last_rendered)
        else:
            self.notify("Primero renderizá el audio (ctrl+r)", severity="warning")
    
    def action_quit(self):
        self.exit()
    
    def action_close_menu(self):
        if self.screen is not self:
            self.pop_screen()


# ── Punto de entrada ──────────────────────────────────────────────

if __name__ == "__main__":
    app = AudioEditorApp()
    app.run()'''

with open("audio_editor_tui.py", "w", encoding="utf-8") as f:
    f.write(tui_code)

print("TUI generada: audio_editor_tui.py")
print("\n" + "="*60)
print("  PASO SIGUIENTE:")
print("  Hace clic en el icono >_ (terminal) en Colab")
print("  y ejecuta:")
print("    python audio_editor_tui.py")
print("="*60)
print("\nAtajos dentro de la TUI:")
print("  Ctrl+P  - Preview del audio editado")
print("  Ctrl+R  - Renderizar audio editado")
print("  Click der. en palabra - Menu contextual")
print("  Ctrl+Q  - Salir")

## Descargar audio editado
Despues de editar y renderizar en la TUI, ejecuta esta celda para descargar.

In [ ]:
# Celda 5: Descargar resultado
from google.colab import files
import glob, os

# Buscar el archivo renderizado mas reciente
wavs = glob.glob("*.wav")
editados = [f for f in wavs if f != 'input_audio.wav']
if editados:
    # Ordenar por fecha de creacion
    editados.sort(key=lambda f: os.path.getctime(f), reverse=True)
    print(f"Descargando: {editados[0]}")
    files.download(editados[0])
else:
    print("No hay archivos editados. Primero corre la TUI y renderiza.")
    print("Si ya renderizaste, busca archivos .wav en el explorador de archivos de Colab.")